   # Modelo de predição Score de crédito


 # 1_fase: Importando bibliotecas

In [ ]:
import pandas as pd 
import numpy as np
import plotly.express as px
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
import seaborn as sns
import matplotlib.pyplot as plt

 # 2_fase: Lendo o DataFrame

 ### *Essa parte tem o objetivo de trazer os dados, ver se estão corretos, e ajudar a facilitar a leitura posteriormente*

 ### Trazendo o df atravéz do pandas e vendo se veio corretamente

In [ ]:
df = pd.read_csv("dados_usados/lending_club_loan_two.csv")

 Vendo as 5 primeiras linhas

In [ ]:
df.head()

,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,...,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,application_type,mort_acc,pub_rec_bankruptcies,address
0,10000.0,36 months,11.44,329.48,B,B4,Marketing,10+ years,RENT,117000.0,...,16.0,0.0,36369.0,41.8,25.0,w,INDIVIDUAL,0.0,0.0,"0174 Michelle Gateway\r\nMendozaberg, OK 22690"
1,8000.0,36 months,11.99,265.68,B,B5,Credit analyst,4 years,MORTGAGE,65000.0,...,17.0,0.0,20131.0,53.3,27.0,f,INDIVIDUAL,3.0,0.0,"1076 Carney Fort Apt. 347\r\nLoganmouth, SD 05113"
2,15600.0,36 months,10.49,506.97,B,B3,Statistician,< 1 year,RENT,43057.0,...,13.0,0.0,11987.0,92.2,26.0,f,INDIVIDUAL,0.0,0.0,"87025 Mark Dale Apt. 269\r\nNew Sabrina, WV 05113"
3,7200.0,36 months,6.49,220.65,A,A2,Client Advocate,6 years,RENT,54000.0,...,6.0,0.0,5472.0,21.5,13.0,f,INDIVIDUAL,0.0,0.0,"823 Reid Ford\r\nDelacruzside, MA 00813"
4,24375.0,60 months,17.27,609.33,C,C5,Destiny Management Inc.,9 years,MORTGAGE,55000.0,...,13.0,0.0,24584.0,69.8,43.0,f,INDIVIDUAL,1.0,0.0,"679 Luna Roads\r\nGreggshire, VA 11650"


 Vendo as ultimas 5 linhas

In [ ]:
df.tail()

,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,...,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,application_type,mort_acc,pub_rec_bankruptcies,address
396025,10000.0,60 months,10.99,217.38,B,B4,licensed bankere,2 years,RENT,40000.0,...,6.0,0.0,1990.0,34.3,23.0,w,INDIVIDUAL,0.0,0.0,"12951 Williams Crossing\r\nJohnnyville, DC 30723"
396026,21000.0,36 months,12.29,700.42,C,C1,Agent,5 years,MORTGAGE,110000.0,...,6.0,0.0,43263.0,95.7,8.0,f,INDIVIDUAL,1.0,0.0,"0114 Fowler Field Suite 028\r\nRachelborough, ..."
396027,5000.0,36 months,9.99,161.32,B,B1,City Carrier,10+ years,RENT,56500.0,...,15.0,0.0,32704.0,66.9,23.0,f,INDIVIDUAL,0.0,0.0,"953 Matthew Points Suite 414\r\nReedfort, NY 7..."
396028,21000.0,60 months,15.31,503.02,C,C2,"Gracon Services, Inc",10+ years,MORTGAGE,64000.0,...,9.0,0.0,15704.0,53.8,20.0,f,INDIVIDUAL,5.0,0.0,"7843 Blake Freeway Apt. 229\r\nNew Michael, FL..."
396029,2000.0,36 months,13.61,67.98,C,C2,Internal Revenue Service,10+ years,RENT,42996.0,...,3.0,0.0,4292.0,91.3,19.0,f,INDIVIDUAL,NaN,0.0,"787 Michelle Causeway\r\nBriannaton, AR 48052"


 ### Criando um dicionario, para falar o que cada uma significa

In [ ]:
colunas_traducao = {
    "loan_amnt": "valor_do_emprestimo",
    "term": "prazo_do_emprestimo_meses",
    "int_rate": "taxa_de_juros",
    "installment": "valor_da_parcela",
    "grade": "classificacao_de_risco",
    "sub_grade": "sub_classificacao_de_risco",
    "emp_title": "cargo_profissao",
    "emp_length": "tempo_de_emprego_anos",
    "home_ownership": "tipo_de_moradia",
    "annual_inc": "renda_anual",
    "verification_status": "status_de_verificacao_da_renda",
    "issue_d": "data_de_emissao_do_emprestimo",
    "loan_status": "status_do_emprestimo",
    "purpose": "finalidade_do_emprestimo",
    "title": "titulo_descricao_do_emprestimo",
    "dti": "relacao_divida_renda",
    "earliest_cr_line": "primeira_linha_de_credito",
    "open_acc": "contas_de_credito_abertas",
    "pub_rec": "registros_publicos_negativos",
    "revol_bal": "saldo_de_credito_rotativo",
    "revol_util": "utilizacao_do_credito_rotativo",
    "total_acc": "total_de_contas_de_credito",
    "initial_list_status": "status_inicial_de_listagem",
    "application_type": "tipo_de_aplicacao",
    "mort_acc": "contas_de_hipoteca",
    "pub_rec_bankruptcies": "registros_de_falencia",
    "address": "endereco"
}

 ### Renomeando as colunas

In [ ]:
df = df.rename(columns = colunas_traducao)

 ### Observando se está certo

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396030 entries, 0 to 396029
Data columns (total 27 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   valor_do_emprestimo             396030 non-null  float64
 1   prazo_do_emprestimo_meses       396030 non-null  object 
 2   taxa_de_juros                   396030 non-null  float64
 3   valor_da_parcela                396030 non-null  float64
 4   classificacao_de_risco          396030 non-null  object 
 5   sub_classificacao_de_risco      396030 non-null  object 
 6   cargo_profissao                 373103 non-null  object 
 7   tempo_de_emprego_anos           377729 non-null  object 
 8   tipo_de_moradia                 396030 non-null  object 
 9   renda_anual                     396030 non-null  float64
 10  status_de_verificacao_da_renda  396030 non-null  object 
 11  data_de_emissao_do_emprestimo   396030 non-null  object 
 12  status_do_empres

 # 3_fase: Analisando e limpando dados

 ### Nessa parte há o objetivo de tirar dados com pouca importancia, ver o que fazer com dados nulos, e organizar para tudo se transformar em dados númericos para ser lido pelo modelo

 ### 3.1: Limpando os dados

 ### Tirando dados que pouca importância

In [ ]:
df = df.drop(["titulo_descricao_do_emprestimo", "cargo_profissao", "status_inicial_de_listagem", "status_de_verificacao_da_renda"], axis=1)

 ### Observando se há dados vazios

In [ ]:
df.isnull().sum()

valor_do_emprestimo                   0
prazo_do_emprestimo_meses             0
taxa_de_juros                         0
valor_da_parcela                      0
classificacao_de_risco                0
sub_classificacao_de_risco            0
tempo_de_emprego_anos             18301
tipo_de_moradia                       0
renda_anual                           0
data_de_emissao_do_emprestimo         0
status_do_emprestimo                  0
finalidade_do_emprestimo              0
relacao_divida_renda                  0
primeira_linha_de_credito             0
contas_de_credito_abertas             0
registros_publicos_negativos          0
saldo_de_credito_rotativo             0
utilizacao_do_credito_rotativo      276
total_de_contas_de_credito            0
tipo_de_aplicacao                     0
contas_de_hipoteca                37795
registros_de_falencia               535
endereco                              0
dtype: int64

 ### olhando se o tipo dos dados são númericos para conseguir completar os dados com medianas, e não perder dados significativos

In [ ]:
df["contas_de_hipoteca"].value_counts()

contas_de_hipoteca
0.0     139777
1.0      60416
2.0      49948
3.0      38049
4.0      27887
5.0      18194
6.0      11069
7.0       6052
8.0       3121
9.0       1656
10.0       865
11.0       479
12.0       264
13.0       146
14.0       107
15.0        61
16.0        37
17.0        22
18.0        18
19.0        15
20.0        13
24.0        10
22.0         7
21.0         4
25.0         4
27.0         3
26.0         2
32.0         2
31.0         2
23.0         2
34.0         1
28.0         1
30.0         1
Name: count, dtype: int64

In [ ]:
df["registros_de_falencia"].value_counts()

registros_de_falencia
0.0    350380
1.0     42790
2.0      1847
3.0       351
4.0        82
5.0        32
6.0         7
7.0         4
8.0         2
Name: count, dtype: int64

In [ ]:
df["utilizacao_do_credito_rotativo"].value_counts()

utilizacao_do_credito_rotativo
0.00      2213
53.00      752
60.00      739
61.00      734
55.00      730
          ... 
146.10       1
109.30       1
108.10       1
115.30       1
37.63        1
Name: count, Length: 1226, dtype: int64

 ### Completando os dados que faltam nessas colunas com medianas considerando todos os valores que ja existem na sua coluna

In [ ]:
df["utilizacao_do_credito_rotativo"] = df["utilizacao_do_credito_rotativo"].fillna(df["utilizacao_do_credito_rotativo"].median())
df["contas_de_hipoteca"] = df["contas_de_hipoteca"].fillna(df["contas_de_hipoteca"].median())
df["registros_de_falencia"] = df["registros_de_falencia"].fillna(df["registros_de_falencia"].median())

 ### Olhando os dados que faltam

In [ ]:
df.isna().sum()

 ### Pegando a coluna que ainda tem dados faltando e aproveitando para mexer em outra que também é data

 ### Olhando como a coluna "tempo_de_emprego_anos" está organizada

In [ ]:
df["tempo_de_emprego_anos"].value_counts()

 ### Criando um dicionario para transformar os dados dela em inteiros

In [ ]:
tempo_emprego = {
"10+ years" : 10,
"2 years" : 2,
"< 1 year" : 0,
"3 years" : 3,
"5 years" : 5,
"1 year" : 1, 
"4 years" : 4, 
"6 years" : 6,
"7 years" : 7 , 
"8 years" : 8,
"9 years" : 9
}

 ### Olhando a outra coluna que também é data e classificada como objeto

In [ ]:
df["prazo_do_emprestimo_meses"].value_counts()

 ### Uma para resolver passei uma função map que passa o dicionario e muda os dados pelas correções que estão no proprio dicionario, e em outro so tirei o "months" e mudei para "int" as respostas

In [ ]:
df["tempo_de_emprego_anos"] = df["tempo_de_emprego_anos"].map(tempo_emprego)
df["prazo_do_emprestimo_meses"] = df["prazo_do_emprestimo_meses"].str.replace("months", "").astype(int)

 ### Olhando se está tudo certo

In [ ]:
df["tempo_de_emprego_anos"].value_counts()

In [ ]:
df["prazo_do_emprestimo_meses"].value_counts()

 ### Agora completo os dados que estavam faltando na coluna "tempo_de_emprego_anos"

In [ ]:
df["tempo_de_emprego_anos"] = df["tempo_de_emprego_anos"].replace(np.nan, df["tempo_de_emprego_anos"].median())

 ### Olho com "sample" para vir os dados aleatoriamente das linhas na coluna e vêr se deu certo

In [ ]:
df["tempo_de_emprego_anos"].sample(20)

 ### Tirando colunas que não sei qual foi o criterio para serem classificadas e meu modelo irá tenter prever essa resposta, então não faz sentido telas

In [ ]:
df = df.drop(["classificacao_de_risco", "sub_classificacao_de_risco"], axis=1)

 ### Olhando se tem mais alguma colunas sem dados

In [ ]:
df.isnull().sum()

 ### Olhando o tipo das colunas agora, para pensar em o que fazer

In [ ]:
df.info()

 ### Criando um laço que separa as colunas que são númericas e as que são textos para ver como está cada

In [ ]:
colunas_numericas = []
colunas_texto = []
for i in df.columns.tolist():
  if df[i].dtype == "int64" or df[i].dtype == "float64":
     colunas_numericas.append(i)
  else:
     colunas_texto.append(i)

In [ ]:
colunas_numericas

In [ ]:
colunas_texto

 ### Tirando algumas coulunas que poderiam ser importantes, mas pesaria muito o processamento para esse teste, em um trabalho importante, com muito mais dados e um bom computador talvez valeria a pena

In [ ]:
df = df.drop(["endereco", "data_de_emissao_do_emprestimo", "primeira_linha_de_credito"], axis=1)

 ### Fazendo o upload desse df agora limpo para ser usado no modelo, codificado e descodificado lá com o mesmo processo que sera usado aqui, mas em outro ambiente para ficar mais facil a decodificação

In [ ]:
df.to_csv("dados_para_modelo.csv", sep=",", encoding="utf-8", index=False)

 ### 3.2: Codificando as colunas objetos para númericas

 ### Separando as colunas que são objetos, exceto a do "status_do_emprestimo" que a maquina irá tentar prever, criando um codificador que fará cada resposta diferente em cada coluna virar número e ainda deixando o nome da coluna que estava no df do pandas, depois de transformar junta com as colunas inteiras que ja tinham no df, fazendo um df_modelo

In [ ]:
colunas_codificadas = df.select_dtypes(include="object").columns.drop("status_do_emprestimo")

codificador = OneHotEncoder(sparse_output=False, drop="first").set_output(transform="pandas")
df_codificado = codificador.fit_transform(df[colunas_codificadas])
df_modelo = pd.concat([df.drop(colunas_codificadas, axis=1), df_codificado], axis=1)

In [ ]:
colunas_codificadas

 ### Observando se está correto

In [ ]:
df_modelo.info()

 ### Transformando separadamente o "status_do_emprestimo"

In [ ]:
df["status_do_emprestimo"].value_counts()

In [ ]:
status_emprestimo = {
"Fully Paid" : 1,
"Charged Off" : 0
}

In [ ]:
df_modelo["status_do_emprestimo"] = df_modelo["status_do_emprestimo"].map(status_emprestimo)

 ### Deixando em primeiro só para não ficar caçando no gráfico de correlação

In [ ]:
df_modelo.set_index("status_do_emprestimo", inplace=True)

In [ ]:
df_modelo.reset_index(inplace=True)

In [ ]:
df_modelo.sample(20)

 # 4_fase: Criando gŕaficos e analisanod

 ### 4.1 Gráfico de correlação

In [ ]:
corr = df_modelo.corr()

grafico_calor = plt.figure(figsize=(30, 8))

sns.heatmap(corr, annot=True, cmap= "coolwarm")

grafico_calor.show

 ### Aqui pode ser visto que não existe nenhum separadamente que tem muita relação com o "status_do_emprestimo", apenas um pouco a "taxa_de_juros"